# Model Evaluation & Benchmark Comparison
### AI-Based Smart Gate Monitoring System
**Author:** Siddharth Dhanush R (Team Lead)
**Objective:** Evaluate baseline `yolov8n.pt` and custom fine-tuned `gate_yolov8n_best.pt` on the same held-out test dataset, computing Precision, Recall, mAP@50, mAP@50-95, per-class metrics, confusion matrices, and inference latency.

## 1. Setup Environment & Load Held-Out Test Set

In [ ]:
!pip install -q ultralytics pandas matplotlib seaborn

import os
import time
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"[INFO] Evaluation Device: {device}")

# Paths to models and test dataset yaml
BASELINE_WEIGHTS = 'yolov8n.pt'
CUSTOM_WEIGHTS = '/content/drive/MyDrive/GateMonitoringModels/gate_yolov8n_best.pt'
DATA_YAML = '/content/drive/MyDrive/GateDataset/data.yaml'

## 2. Evaluate Baseline YOLOv8n (Experiment A)

In [ ]:
print("[INFO] Evaluating Baseline YOLOv8n...")
baseline_model = YOLO(BASELINE_WEIGHTS)
baseline_metrics = baseline_model.val(data=DATA_YAML, split='test', device=device)

b_map50 = baseline_metrics.box.map50
b_map50_95 = baseline_metrics.box.map
b_p = baseline_metrics.box.mp
b_r = baseline_metrics.box.mr
print(f"[BASELINE] Precision: {b_p:.4f} | Recall: {b_r:.4f} | mAP50: {b_map50:.4f} | mAP50-95: {b_map50_95:.4f}")

## 3. Evaluate Custom Fine-Tuned Model (Experiment B)

In [ ]:
if os.path.exists(CUSTOM_WEIGHTS):
    print("[INFO] Evaluating Custom Fine-Tuned Model...")
    custom_model = YOLO(CUSTOM_WEIGHTS)
    custom_metrics = custom_model.val(data=DATA_YAML, split='test', device=device)
    
    c_map50 = custom_metrics.box.map50
    c_map50_95 = custom_metrics.box.map
    c_p = custom_metrics.box.mp
    c_r = custom_metrics.box.mr
    print(f"[CUSTOM] Precision: {c_p:.4f} | Recall: {c_r:.4f} | mAP50: {c_map50:.4f} | mAP50-95: {c_map50_95:.4f}")
else:
    print(f"[INFO] Custom weights not found at {CUSTOM_WEIGHTS}. Run Notebook 03 first.")

## 4. Benchmark Inference Speeds

In [ ]:
# Benchmark latency on 100 iterations
dummy_input = np.zeros((640, 640, 3), dtype=np.uint8)

def benchmark(m):
    # Warmup
    for _ in range(10):
        _ = m.predict(dummy_input, device=device, verbose=False)
    lats = []
    for _ in range(50):
        t0 = time.perf_counter()
        _ = m.predict(dummy_input, device=device, verbose=False)
        lats.append((time.perf_counter() - t0) * 1000.0)
    return np.mean(lats), 1000.0 / np.mean(lats)

b_lat, b_fps = benchmark(baseline_model)
print(f"[BASELINE SPEED] Latency: {b_lat:.2f} ms | FPS: {b_fps:.1f}")

## 5. Comparative Summary & Export

In [ ]:
summary_data = {
    'Model': ['Baseline YOLOv8n', 'Custom Gate YOLOv8n'],
    'Weights': ['yolov8n.pt', 'gate_yolov8n_best.pt'],
    'Precision': [round(b_p, 4), 'TBD'],
    'Recall': [round(b_r, 4), 'TBD'],
    'mAP50': [round(b_map50, 4), 'TBD'],
    'mAP50-95': [round(b_map50_95, 4), 'TBD'],
    'Latency (ms)': [round(b_lat, 2), 'TBD'],
    'FPS': [round(b_fps, 1), 'TBD']
}
df = pd.DataFrame(summary_data)
print(df.to_string(index=False))
# Export to CSV
df.to_csv('model_comparison_results.csv', index=False)